[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Why Gradient Boosting


## What you will be able to do

Write gradient boosting as a loop of small trees, each fitted to the errors left by the trees before
it, and then train the same kind of model in a few lines of LightGBM and of XGBoost.

## The idea

### The problem

A weather service keeps a table of 20,000 days at 400 stations: where each station stands, the day
of the year, the cloud, the wind, the snow on the ground, and the day's lowest temperature. The
coldest nights come from a combination. A clear sky lets the ground lose heat, calm air lets the cold
settle, and a valley collects it, so cloud, wind, season and terrain matter together, each one by an
amount that depends on the others.

The **scikit-learn, Deep Dive** guide fitted straight lines and forests. A straight line adds up each
feature with one weight, so it cannot say that wind matters more on a clear night than on a cloudy
one. A shallow decision tree can, in coarse steps, and a deep one learns the noise. A random forest
averages hundreds of deep trees grown independently, and each of them still learns the whole answer
on its own.

Gradient boosting divides the work differently. Its trees are small and come one after another,
each fitted to what the trees before it got wrong.

### A formal definition

> **Gradient boosting** builds a model as a sum of many small decision trees, added one at a time.
> Each new tree is fitted to the **residuals**, the differences between the answers and what the
> trees so far predict, and its predictions are scaled by a **learning rate** before they are added.
> For a loss other than squared error, the residuals are replaced by the negative gradient of the
> loss, which gives the method its name.

### Why it works that way

A tree with a few splits can say that clear January nights are colder, and little else. The next
tree sees only what that left unexplained, so it can find what the first one missed, such as the
extra cold of a calm valley, and after hundreds of rounds the sum describes combinations no small
tree could. The learning rate makes every step partial, so each tree corrects part of the remaining
error and leaves the rest to later trees, which the **Learning Rate and Trees** notebook tests.
Adding trees
keeps reducing the training error, so at some point the trees start fitting noise, and a validation
set decides how many to add.

LightGBM and XGBoost add refinements of their own. Both sort each feature's values into
bins once, so a split is chosen by scanning a few hundred bins instead of every row, both send a row
whose value is missing down whichever side of a split works better, and both accept a column of
categories. They differ in how a tree grows and in their defaults, which later notebooks compare.

### Where this shows up

Boosted trees are a common first model for a table of mixed features, and scikit-learn has its own,
`HistGradientBoostingRegressor`, built on the same binned features. The **Early Stopping** notebook
lets a validation set choose the number of trees, and the Visual Handbook covers the statistics.

### What this notebook covers

- the made-up table of station-days, split by station
- one small tree, and boosting written by hand as a loop of small trees
- the same model in LightGBM and in XGBoost
- every model compared on the test stations
- three mistakes from these first steps

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
from sklearn.tree import DecisionTreeRegressor

x = np.arange(10).reshape(-1, 1)
y = (x.ravel() - 4.5) ** 2
prediction = np.full(10, y.mean())
for step in range(1, 4):
    tree = DecisionTreeRegressor(max_depth=1).fit(x, y - prediction)
    prediction = prediction + 0.5 * tree.predict(x)
    print(step, round(((y - prediction) ** 2).mean(), 2))
```

```
1 40.8
2 26.72
3 19.0
```

Every prediction starts at the mean of the U-shaped answers, a mean squared error of 52.8. Each tree
has one split and is fitted to `y - prediction`, what the predictions so far get wrong, and adding
half of its answer lowers the error at every step.

## Setup

Two libraries for the table, the two boosting libraries, and four parts of scikit-learn.

- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `DecisionTreeRegressor` is the small tree the hand-written loop is made of
- `LinearRegression` and `RandomForestRegressor` are the models to compare with
- `mean_squared_error` scores every model on the test stations

`station_days` makes 20,000 rows of daily weather at 400 weather stations in Norway. The stations
and their days are made up, and a generator started from a fixed seed draws them, so the table is the
same on every run. Each row has the station's latitude, elevation in meters, distance from the coast
in kilometers and terrain, the day of the year, the cloud cover from 0 to 1, the wind in meters a
second, the snow on the ground in centimeters, and `tmin`, the day's lowest temperature in degrees
Celsius. `terrain` is a pandas category, and `snow_cm` is missing at stations that have no snow
sensor. `FEATURES` names the eight columns a model is given.

The cell splits the stations into 300 for training, 50 for validation and 50 for testing, keeping
all of a station's days on one side, because every station has a warmth or coldness of its own, and a
model that had seen other days of a test station would know it. This notebook uses the training and test stations, and the
**Early Stopping** notebook uses the validation stations.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### The station-days table

In [2]:
print(days.shape)
print(days.dtypes)

(20000, 10)
station          object
latitude        float64
elevation_m       int64
coast_km        float64
terrain        category
day_of_year       int64
cloud_cover     float64
wind_ms         float64
snow_cm         float64
tmin            float64
dtype: object


In [3]:
days.head()

,station,latitude,elevation_m,coast_km,terrain,day_of_year,cloud_cover,wind_ms,snow_cm,tmin
0,ST0001,60.33,161,50.3,inland,173,0.08,0.7,NaN,5.3
1,ST0001,60.33,161,50.3,inland,248,0.36,2.3,NaN,8.8
2,ST0001,60.33,161,50.3,inland,86,0.35,2.5,NaN,-5.7
3,ST0001,60.33,161,50.3,inland,102,0.59,9.2,NaN,-0.3
4,ST0001,60.33,161,50.3,inland,252,0.53,5.3,NaN,8.6


Every station has 50 rows, one for each of its days, and the four columns that describe the station
repeat on each of them. `terrain` is a category, and `snow_cm` holds `NaN` where a station has no
snow sensor, as the first station has none.

In [4]:
print("rows with no snow reading:", round(days["snow_cm"].isna().mean(), 3))
print("lowest temperature, mean and spread:", round(days["tmin"].mean(), 2), round(days["tmin"].std(), 2))

rows with no snow reading: 0.315
lowest temperature, mean and spread: -1.45 6.63


31.5% of the rows have no snow reading. The day's lowest temperature averages −1.45 degrees with a
standard deviation of 6.63, so a model that predicted the mean every day would have a mean squared
error of about 6.63², or 44, the number every model below has to beat.

### A small tree on its own

scikit-learn's tree needs every column as a number, so `terrain` becomes five columns of 0 and 1
with `pd.get_dummies`. Its trees accept a missing value, so `snow_cm` stays as it is.

In [5]:
def as_numbers(rows):
    return pd.get_dummies(rows[FEATURES], columns=["terrain"], dtype=float)

X_train, X_test = as_numbers(train), as_numbers(test)
y_train, y_test = train["tmin"], test["tmin"]

print("predict the mean:", round(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())), 3))
tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
print("one tree of depth 3:", round(mean_squared_error(y_test, tree.predict(X_test)), 3))

predict the mean: 44.918
one tree of depth 3: 12.201


Predicting the training mean for every test day leaves a mean squared error of 44.918. One tree of
depth 3, which has at most eight leaves and so at most eight answers, brings it down to 12.201.

### Boosting written by hand

The loop starts every prediction at the training mean. Each round fits a tree of depth 3 to the
residuals, `y_train - prediction`, and adds a tenth of what it predicts, for the training rows and
for the test rows alike.

In [6]:
prediction_train = np.full(len(y_train), y_train.mean())
prediction_test = np.full(len(y_test), y_train.mean())

for round_number in range(1, 301):
    tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train - prediction_train)
    prediction_train += 0.1 * tree.predict(X_train)
    prediction_test += 0.1 * tree.predict(X_test)
    if round_number in (1, 10, 50, 100, 200, 300):
        print(round_number, "training", round(mean_squared_error(y_train, prediction_train), 3),
              "| test", round(mean_squared_error(y_test, prediction_test), 3))

hand_test_loss = mean_squared_error(y_test, prediction_test)

1 training 37.757 | test 38.885
10 training 13.644 | test 14.572
50 training 3.283 | test 3.941
100 training 2.273 | test 2.934
200 training 1.852 | test 2.671
300 training 1.672 | test 2.61


The first round removes little of the error, because only a tenth of its tree's prediction is added.
By round 50 the test loss is 3.941, already below the single tree's 12.201, and by round 300 it is
2.61. The training loss keeps falling further, to 1.672, and the gap between the two is what the trees
have fitted to the training stations alone.

### The same model in LightGBM and XGBoost

Both libraries take the table as it is, the category and the missing values included, so `FEATURES`
goes in without `get_dummies`. `n_estimators` is the number of trees and `learning_rate` the share of
each one that is added, the 0.1 of the loop above. `verbose=-1` stops LightGBM printing a log of
what it is doing.

In [7]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], y_train)
print("LightGBM:", round(mean_squared_error(y_test, lightgbm_model.predict(test[FEATURES])), 3))

xgboost_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=0)
xgboost_model.fit(train[FEATURES], y_train)
print("XGBoost:", round(mean_squared_error(y_test, xgboost_model.predict(test[FEATURES])), 3))

LightGBM: 2.542
XGBoost: 2.618


Both libraries reach about the loss of the hand-written loop, LightGBM 2.542 and XGBoost 2.618, with
the same number of trees and the same learning rate. Their trees are larger than the loop's, since
LightGBM grows a tree to 31 leaves and XGBoost to a depth of 6 unless told otherwise, which the **Tree
Shape** notebook compares.

### Every model on the test stations

The cell fits a linear model and a random forest to the same training stations, for comparison. The
linear model gets `snow_cm` filled with the training median, since it cannot take a missing value,
and a `season` column, the cosine of the day of the year, since a straight line in the day number
cannot describe a year that is cold at both ends.

In [8]:
def for_linear(rows):
    numbers = as_numbers(rows).fillna({"snow_cm": train["snow_cm"].median()})
    return numbers.assign(season=np.cos(2 * np.pi * (rows["day_of_year"] - 20) / 365))

linear = LinearRegression().fit(for_linear(train), y_train)
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=0).fit(X_train, y_train)

scores = {
    "linear model": mean_squared_error(y_test, linear.predict(for_linear(test))),
    "random forest": mean_squared_error(y_test, forest.predict(X_test)),
    "boosting by hand": hand_test_loss,
    "LightGBM": mean_squared_error(y_test, lightgbm_model.predict(test[FEATURES])),
    "XGBoost": mean_squared_error(y_test, xgboost_model.predict(test[FEATURES])),
}
for name, score in scores.items():
    print(f"{name:17} {score:.3f}")

linear model      3.730
random forest     3.071
boosting by hand  2.610
LightGBM          2.542
XGBoost           2.618


The linear model, even with a column for the season, has the largest error, 3.73. The cold of a
clear, calm valley night comes from conditions that multiply each other's effect, and a sum of
weighted columns has no way to multiply. The forest does better, 3.071, and the three boosted models
do best, from 2.542 to 2.618. The differences among those three come from one split of the stations
and one seed, which is too little to rank them.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/01-why-gradient-boosting-solutions.ipynb).

**1.** Run the hand-written loop again with a learning rate of 0.3, and print the test loss after 10, 50 and 300 rounds to see whether a larger step helps.

In [9]:
# your code here

**2.** Run the loop with trees of depth 1 in place of 3, for 300 rounds, and print the test loss.

In [10]:
# your code here

**3.** Train LightGBM with 50 trees and with 1,000 trees, at a learning rate of 0.1, and print the test loss of each beside the 300-tree model's.

In [11]:
# your code here

**4.** Train an XGBoost model with `max_depth=3`, the depth of the hand-written loop's trees, for 300 rounds at 0.1, and print its test loss.

In [12]:
# your code here

**5.** Make a one-row table for a valley station at latitude 69.0, 250 meters up and 60 kilometers from the coast, on day 15 with a cloud cover of 0, a wind of 0.5 meters a second and 40 centimeters of snow, and print what `lightgbm_model` predicts for it. Give `terrain` the same categories as the training table.

In [13]:
# your code here

## Common errors

### `ValueError: pandas dtypes must be int, float or bool.`

A table read from a file holds text, and `terrain` arrives as text rather than as a category.

In [14]:
as_text = train[FEATURES].astype({"terrain": str})
print(as_text["terrain"].dtype)
lgb.LGBMRegressor(verbose=-1).fit(as_text, y_train)

object


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: terrain: object

LightGBM takes numbers, booleans and pandas categories, and the message names the column that is none
of these, `terrain: object`, where `object` is pandas' name for a column of Python strings.

### `ValueError: DataFrame.dtypes for data must be int, float, bool or category.`

In [15]:
xgb.XGBRegressor().fit(as_text, y_train)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:terrain: object

XGBoost's message names the same column at its end, `Invalid columns:terrain: object`. It also
mentions `enable_categorical`, a setting that XGBoost 3.4 turns on by itself for a pandas category,
which is why the worked examples fit `train[FEATURES]` without it. The fix for both libraries is to
make the column a category, with the same categories the training table has.

In [16]:
fixed = as_text.astype({"terrain": pd.CategoricalDtype(TERRAINS)})
print(fixed["terrain"].dtype)
print("LightGBM fitted:", lgb.LGBMRegressor(verbose=-1).fit(fixed, y_train).n_features_in_, "features")
print("XGBoost fitted:", xgb.XGBRegressor().fit(fixed, y_train).n_features_in_, "features")

category
LightGBM fitted: 8 features
XGBoost fitted: 8 features


### No error, and a loss that grows after ten rounds

In [17]:
wrong_train = np.full(len(y_train), y_train.mean())
for round_number in range(1, 51):
    tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
    wrong_train += 0.1 * tree.predict(X_train)
    if round_number in (1, 5, 10, 20, 50):
        print(round_number, "training", round(mean_squared_error(y_train, wrong_train), 3))

1 training 37.78
5 training 20.441
10 training 14.191
20 training 53.101
50 training 581.123


Each tree here is fitted to `y_train`, the whole answer, instead of to the residuals, so every round
adds another tenth of the answer to predictions that already hold some of it. The loss falls while the
sum is still short of the answer, to 14.191 by round 10, and then climbs far above the first round's
37.78, to 581.123 by round 50, with no error to say why. Fit each tree to `y_train - prediction`, as the worked
examples do.

## Recap

- You can write gradient boosting as a loop that fits each small tree to the residuals and adds a
  share of its prediction.
- You can train the same kind of model with `LGBMRegressor` and `XGBRegressor`, giving each the table
  with its category and its missing values.
- You can compare boosted trees with a linear model and a forest on stations none of them trained on.
- You can make a text column a pandas category before either library sees it.

## What is next

The **scikit-learn Interface** notebook looks at the classes used here, their settings and defaults,
and how they fit into a scikit-learn workflow.

---

[LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [The scikit-learn Interface](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/02-the-scikit-learn-interface.ipynb) &#8594;
